<a href="https://colab.research.google.com/github/XaVIIIerg/deep-learning-nlp-coursework/blob/main/MSE_AdvNLP_Lab4_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![MSE Logo](https://moodle.msengineering.ch/pluginfile.php/1/core_admin/logo/0x150/1643104191/logo-mse.png)

# AdvNLP Lab 4: RAG - Information Retrieval and Reranking

**Objectives:** We build the retrieval part of a RAG system and compare performance of classic KNN retrieval with additional cross encoder reranking. Eventually, we write two prompts for generation and test it on a LLM.

**Useful documentation:** Since you'll use LangChain for this assignment, [their documentation](https://python.langchain.com/docs/introduction/) might be helpful.

## Setup

First, we need to install the required packages for this assignment.

In [ ]:
#!pip install pandas langchain-community langchain-huggingface faiss-cpu --quiet -langchain

In [ ]:
!pip install pandas langchain-community langchain-huggingface faiss-cpu --quiet langchain-classic

import pandas as pd
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


We will use a preselection of the [DRAGONBall Dataset](https://github.com/OpenBMB/RAGEval) as a basis for this assignment and load a subset of documents. These documents will be the external knowledge base of the RAG system to be stored in a vector data base. Explicit chunking is not needed for our case, since the size of information pieces is suitable as chunks.

As we see each document consists of a unique ID and the actual content.

In [ ]:
documents = pd.read_csv('docs.csv', index_col=0)
documents

,content
id,
40,Acme Government Solutions is a government indu...
41,Entertainment Enterprises Inc. is an entertain...
42,"Advanced Manufacturing Solutions Inc., establi..."
43,"EcoGuard Solutions, established on April 15, 2..."
44,"Green Fields Agriculture Ltd., established on ..."
...,...
211,Hospitalization Record:\n\nBasic Information:\...
212,**Hospitalization Record**\n\n**Basic Informat...
213,Hospitalization Record\n\nBasic Information:\n...


The main goal of the assignment is to evaluate the retrieval component of the RAG system. To this end, a dataset of queries is provided, which can be used to retrieve matching documents. For each query an array of relevant documents is given specified by one or more IDs from the beforementioned vector data base. We can thus evaluate whether the correct documents were found by the retriever.

In [ ]:
queries = pd.read_csv('queries.csv', index_col=0)
queries['ground_truth_doc_ids'] = queries['ground_truth_doc_ids'].apply(lambda x: x.split(';'))
queries

,query,ground_truth_doc_ids
query_id,,
2286,When was Sparkling Clean Housekeeping Services...,[64]
2433,How did HealthPro Innovations' strategic partn...,[54]
6266,According to the hospitalization records of Br...,[212]
4499,"According to the judgment of Norwood, Unionvil...",[124]
2448,Based on HealthLife Solutions' 2020 corporate ...,[73]
...,...,...
2186,How did the severe drought in August 2018 lead...,[65]
3251,Compare the large-scale financing activities o...,"[58, 55]"
2268,How did CleanCo Housekeeping Services' investm...,[47]


## 1. Recall@N

**1a)** We will evaluate the retrieval by comparing the retrieved documents with the ground truth documents assigned to the query. For that, we will use the Recall@N metric. Please describe in 1-2 sentences how we can interpret this metric in our case.

**Your Answer:** Recall@N measures what fraction of queries successfully retrieve the target document within the top-N results, where the N represents how many top results we consider. For each query, the target document is either in the top-N results (success = 1) or it isn’t (failure = 0).


The formula is *Recall@N = (Number of queries with target in top-N) / (Total queries)*.

**1b)** Now, please implement the Recall@N metric and test it with the following code.

In [ ]:
def recall_at_n(retrieved_docs, relevant_doc_ids, n):
    """
    Calculate Recall@N.

    Parameters:
    - retrieved_docs: Sorted list of retrieved documents as LangChain Document objects
    - relevant_doc_ids: List of relevant document IDs
    - n: Number of top documents to consider

    Returns:
    - Recall@N
    """
    sum=0
    for doc in retrieved_docs[:n]:
        if doc.metadata['id'] in relevant_doc_ids:
            sum+=1
    return sum/n


In [ ]:
### Test

recall_at_n(
    [Document(page_content='', metadata={'id': str(id)}) for id in range(10)],
    ['0', '1', '20'],
    3
)

0.6666666666666666

**Your Answer**:
For the given test cases the Recall@N is 0.66 = 2/3.

2 documents out of 3 are relevant for the query.

## 2. Embedding Model

**2a)** Next, each document has to be converted to an embedding representing the semantic meaning of the document. For this, please use the embedding model `sentence-transformers/all-MiniLM-L6-v2` from HuggingFace and answer indicate characteristic information about the model:

In [ ]:
from langchain_core import embeddings
model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.embed_documents(documents['content'])
print(len(embeddings)) #nombre de docs
print(len(embeddings[0])) # embedding size

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

108
384


**Your Answers:**

Embedding Size: 384

Number of Parameters: 22.7M according to HuggingFace

Maximum Sequence Length: 128 tokens according to HuggingFace

## 3. Vector Store

**3a)** We now use LangChain to create a FAISS vector store and embed the documents with the above-mentioned embedding model. Please load the documents again but this time with a Loader object from LangChain. Eventually, print the number of documents in the vector store.

In [ ]:
from langchain_community.document_loaders import DataFrameLoader
#documents['id'] = documents.index.astype(str)
loader = CSVLoader('docs.csv', source_column="content", metadata_columns=["id"])
docs = loader.load()
vector_store = FAISS.from_documents(docs, model)
print(vector_store.index.ntotal)

108


**Your Answer**:

The correct number of documents is 108

**3b)** Retrieve now the Top-3 documents for this query: "According to the hospitalization records of Bridgewater General Hospital, summarize the present illness of J. Reyes." and print the documents' ID and L2 distance.

In [ ]:
new_query = "According to the hospitalization records of Bridgewater General Hospital, summarize the present illness of J. Reyes."
docs_found = vector_store.similarity_search_with_score(new_query, k=3)
for doc in docs_found :
  print("document " + doc[0].metadata['id'] + " distance " + str(doc[1]))



document 212 distance 0.73022735
document 213 distance 0.9898045
document 210 distance 1.0050285


**Your Answers**:

The Top-3 IDs and similarities are 212 with 0.73, doc 213 with 0.98, doc 210 with 1.00

**3c)** Please check and show if a suitable document is found for the query in the Top-3 retrieved documents and show the relevant ones.

In [ ]:
#print(type(queries))
#result = queries[queries["query"]=="According to the hospitalization records of Bridgewater General Hospital, summarize the present illness of J. Reyes."]
#print(result)
#print(result["ground_truth_doc_ids"].values)
print(docs_found[0][0])



page_content='content: **Hospitalization Record**

**Basic Information:**
Name: J. Reyes
Gender: Male
Age: 52
Ethnicity: Hispanic
Marital Status: Married
Occupation: Construction Worker
Address: 22, Sunnyvale street, Bridgewater
Admission Time: 7th, September
Record Time: 8th, September
Historian: Self
Hospital Name: Bridgewater General Hospital

**Chief Complaint:**
Persistent joint pain and morning stiffness for 6 months

**Present Illness:**
Onset: The symptoms began insidiously 6 months ago, initially noticed while working at a construction site. Gradual onset with morning stiffness in the fingers and wrists.
Main Symptoms: Morning stiffness, arthritis affecting hands, feet, wrists, ankles, and temporomandibular joints. Pain characterized as dull and persistent, worsens with activity and improves with rest.
Accompanying Symptoms: Joint deformities in the hands, fatigue, intermittent fever, and weight loss of approximately 5 kg over the past few months.
Diagnosis and Treatment Histo

In [ ]:
print(docs_found[1][0])


page_content='content: Hospitalization Record

Basic Information:
Name: K. Ramos
Gender: Female
Age: 82
Ethnicity: Caucasian
Marital Status: Widowed
Occupation: Retired
Address: 21, Greenfield Street, Windsor
Admission Time: 26th October
Record Time: 26th October, 10:00 AM
Historian: S. Martinez, MD
Hospital Name: Windsor General Hospital

Chief Complaint:
Severe bone pain and difficulty in walking for 3 days.

Present Illness:
The symptoms started suddenly 3 days ago while at home, without any clear prodromal symptoms or identifiable triggers. K. Ramos reports severe, throbbing pain localized in the left femur. The pain has been continuous, rated as an 8 out of 10 in severity, and is aggravated by walking or any weight-bearing activities. Local redness and swelling are also noted. Fever of 38.5°C (101.3°F) and general fatigue were observed, coinciding with the onset of bone pain. Patients visited her primary care physician two days ago, who ordered an X-ray and referred her to this ho

In [ ]:

print(docs_found[2][0])

page_content='content: Hospitalization Record

Basic Information:
Name: J. Alvarez
Gender: Female
Age: 83
Ethnicity: Hispanic
Marital Status: Widowed
Occupation: Retired
Address: 23, Yarmouth Street, Clearwater
Admission Time: 21st, August
Record Time: 21st, August
Historian: Self
Hospital Name: Clearwater General Hospital

Chief Complaint:
Repeated respiratory infections, palpitations, and shortness of breath for 2 months.

Present Illness:
Onset:
Symptoms started 2 months ago with no specific triggering event. Initially, patient noticed mild shortness of breath and occasional palpitations.

Main Symptoms:
Patient reports progressive shortness of breath, frequent palpitations, and multiple episodes of respiratory infections over the past two months. Symptoms worsen with physical exertion.

Accompanying Symptoms:
Cyanosis especially during infections; occasional lightheadedness and fatigue; no chest pain or syncope.

Diagnosis and Treatment History:
Patient visited local clinic where s

**Your Answer:** We can see that only the document with the ID 212 is about J.Reyes. The others apply to other people.

## 4. Vector Store Evaluation

**4a)** Now, please search for each of the queries the most relevant documents in the vector store, and calculate Recall@N given the respective assigned ground truth document IDs. To aggregate the results over all queries, we will calculate the mean. This assessment needs to be carried out for different values of $N$ i.e.,  $N \in \{ 1, 3, 5, 25\}$.

In [ ]:
N = [1, 3, 5, 25]
relevant_means = {}

for n in N:
  relevant_means[n] = 0
  for index, row in queries.iterrows():
    docs_found = vector_store.similarity_search_with_score(row['query'], k=n)
    relevant_means[n] += recall_at_n(retrieved_docs=[doc for doc,_ in docs_found], relevant_doc_ids=row['ground_truth_doc_ids'], n=n)
  relevant_means[n] /= len(queries)

print(relevant_means)

0.82
0.3566666666666666
0.22999999999999965
0.05520000000000004


**Your Answers**:

* The average Recall@1 is: 0.82
* The average Recall@3 is: 0.35
* The average Recall@5 is:0.22
* The average Recall@25 is: 0.05

**4b)** When looking at the four calculated Recall@N scores, what do you observe and how can you explain this?

**Your Answer:** The Higher the N, the lower the average recall. This is easily explained by the fact that there is very rarely more than 3 ground_truth documents for a querie, so naturally when we retrieve 25 documents there will be at least 22 of them not in the ground truth, lowering the average recall at N.

## 5. Cross Encoder

**5a)** We want to use a cross encoder model to rerank the retrieved documents. Please, describe in a few sentences how a new document order can be determined using a cross encoder.

**Your Answer:**
We concatenate the text and the query to calculate a score of similarity, to show how the query and the text are linked. We apply this technique over the retrieved documents. With scores of similarity, we can rerank the documents.

**5b)** Now again, we want to calculate Recall@N for all queries and the same $N$ as before. This time, we want to rerank the Top-25 retrieved documents using the cross encoder model `BAAI/bge-reranker-base`. Please, implement this using LangChain components and report the average Recall@N for $N \in \{ 1, 3, 5, 25\}$.

In [ ]:
N_values = [1, 3, 5, 25]
relevant_means = {}
cross_encoder = HuggingFaceCrossEncoder(model_name='BAAI/bge-reranker-base')
reranker = CrossEncoderReranker(model=cross_encoder)

for n in N_values:
  total_recall_at_n = 0
  for index, row in queries.iterrows():
    initial_retrieved_docs_with_scores = vector_store.similarity_search_with_score(row['query'], k=25)
    initial_docs = [doc for doc, _ in initial_retrieved_docs_with_scores]

    reranked_docs = reranker.compress_documents(initial_docs, row['query'])

    n_reranked_docs = reranked_docs[:n]

    total_recall_at_n += recall_at_n(retrieved_docs=n_reranked_docs,relevant_doc_ids=row['ground_truth_doc_ids'],n=n)
  relevant_means[n] = total_recall_at_n / len(queries)

print(relevant_means)

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

{1: 0.9, 3: 0.43666666666666676, 5: 0.2619999999999996, 25: 0.05240000000000004}


**Your Answers**:

* The average Recall@1 is: 0.9
* The average Recall@3 is: 0.44
* The average Recall@5 is: 0.26
* The average Recall@25 is: 0.05

**5c)** What do you observe when you compare the Recall@N scores after reranking with the scores without reranking? Write 1-2 sentences about this and why this might happen.

**Your Answer:**  
The results are better with the new reranking. That can be explained by the fact that cross encoding manages simultaneously the query and the document in the same model to capture their context. We retrieve the most relevant documents before generating responses.

## 6. Generation

**6a)** After improving the retrieval part of the RAG system, we want to finally generate an answer for our query. Please, retrieve the most relevant document for the query "How much funding did HealthPro Innovations raise in February 2021?" and print its ID. Then please write the instruction message of a prompt to answer this query including all necessary elements before running it using your favourite LLM (ChatGPT GPT-4o, etc.). Please paste the answer from the model and indicate which model you used.

In [ ]:
query6a = "How much funding did HealthPro Innovations raise in February 2021?"
most_relevant_doc = vector_store.similarity_search_with_score(query6a, k=1)
print(most_relevant_doc[0][0].metadata['id'])
print(most_relevant_doc[0][0].page_content)

54
content: HealthPro Innovations, established in September 2009, is a publicly traded healthcare company based in New York, specializing in the development and sale of innovative healthcare solutions.
In 2021, HealthPro Innovations experienced several significant events that had a profound impact on its financial performance and market position. Firstly, in February 2021, the company conducted a large-scale financing activity, raising $150 million in funds. This financing activity strengthened the company's financial strength and provided support for its expansion and development. One of the sub-events that contributed to this financing activity was the strategic partnership formed with a leading healthcare provider in January 2021. This partnership aimed to expand HealthPro Innovations' market reach and gain access to new customers, ultimately resulting in increased sales and revenue potential.
In March 2021, HealthPro Innovations successfully launched a new line of innovative health

**Your Prompt:** You are an assistant answering questions

Instructions :
-  use only information coming from the context

Context : HealthPro Innovations, established in September 2009, is a publicly traded healthcare company based in New York, specializing in the development and sale of innovative healthcare solutions.
In 2021, HealthPro Innovations experienced several significant events that had a profound impact on its financial performance and market position. Firstly, in February 2021, the company conducted a large-scale financing activity, raising $150 million in funds. This financing activity strengthened the company's financial strength and provided support for its expansion and development. One of the sub-events that contributed to this financing activity was the strategic partnership formed with a leading healthcare provider in January 2021. This partnership aimed to expand HealthPro Innovations' market reach and gain access to new customers, ultimately resulting in increased sales and revenue potential.
In March 2021, HealthPro Innovations successfully launched a new line of innovative healthcare solutions, offering advanced features and improved patient outcomes. This product launch stimulated customer interest and generated additional revenue streams for the company. Additionally, in April 2021, HealthPro Innovations acquired a promising research and development startup specializing in cutting-edge medical technology. This acquisition enhanced the company's research capabilities, accelerating the development of new and improved healthcare solutions.
Furthermore, in June 2021, HealthPro Innovations expanded its operations into key international markets, establishing partnerships with local distributors and healthcare providers. This expansion increased the company's market share, revenue, and global presence. In August 2021, the company received an industry award for its outstanding innovation and contribution to the healthcare sector. This award enhanced HealthPro Innovations' brand reputation, attracting potential customers, investors, and partners.
In September 2021, HealthPro Innovations achieved a major research breakthrough, resulting in a significant advancement in the effectiveness and efficiency of its healthcare solutions. This breakthrough propelled the company ahead of competitors, increased market demand, and had the potential for higher sales. In November 2021, HealthPro Innovations obtained a crucial regulatory approval for one of its flagship products, allowing for expanded market access and increased sales potential. This approval boosted confidence among customers and investors, driving sales growth and enhancing the company's market position.
These events had a direct impact on several financial indicators for HealthPro Innovations in 2021. The company reported an operating income of $250 million, influenced by market demand and changes in product prices. The net profit for the year was $40 million, influenced by the operating income, cost control, and non-recurring gains and losses. HealthPro Innovations' total assets stood at $500 million, influenced by asset acquisition, disposal, and revaluation. The company's total liabilities amounted to $200 million, influenced by new debt issuance, debt repayment, and debt restructuring. Shareholder equity reached $300 million, influenced by the net profit, dividend distribution, and capital reserves. The company generated a cash flow of $80 million, influenced by operating activities, investment activities, and financing activities. The debt ratio was 40%, reflecting the company's debt level, while the debt to assets ratio was 0.4, indicating the company's financial leverage. The return on equity was 13.33%, reflecting the operational efficiency of shareholder equity.
Looking ahead, HealthPro Innovations aims to implement a comprehensive financial strategy focused on cost control and optimizing capital operations to achieve sustainable growth and profitability. The company plans to invest heavily in research and development of new healthcare solutions, expand into emerging markets, and enhance its manufacturing capabilities. HealthPro Innovations has identified major financial risks such as changes in healthcare regulations, intense competition, and currency fluctuations. The company will closely monitor these risks and implement appropriate risk management strategies to mitigate their impact on financial performance.
In conclusion, HealthPro Innovations has experienced significant events in 2021 that have shaped its financial performance and market position. Through strategic partnerships, product launches, acquisitions, international expansion, and regulatory approvals, the company has strengthened its presence in the healthcare industry. With a focus on financial strategy, investment plans, and risk management, HealthPro Innovations is well-positioned for future growth and success.
The company has been listed on the New York Stock Exchange since its establishment, positioning itself as a key player in the healthcare industry. HealthPro Innovations operates in an industry that is constantly evolving, and as such, it has demonstrated its commitment to compliance and regulatory updates.
In January 2021, HealthPro Innovations took proactive steps to comply with new regulations, particularly in the area of data protection. The company implemented enhanced data protection measures in accordance with updated healthcare data privacy laws. This event ensures that HealthPro Innovations remains compliant with evolving laws, safeguarding sensitive data and protecting the interests of the company and its shareholders.
In March 2021, HealthPro Innovations made a strategic move to acquire MedTech Solutions, a leading provider of medical technology software solutions. This acquisition aimed to expand the company's portfolio of healthcare solutions and enhance its market position. The acquisition of MedTech Solutions not only increases HealthPro Innovations' market share but also broadens its product offerings, leading to potential revenue growth and improved competitiveness in the healthcare industry.
In July 2021, HealthPro Innovations made a significant breakthrough in the field of diagnostic technology. The company launched a state-of-the-art AI-powered diagnostic device that revolutionizes the accuracy and efficiency of medical diagnoses. This innovative product attracted significant attention from healthcare providers and professionals, positioning HealthPro Innovations as an industry leader in diagnostic technology. This event has the potential to increase sales and market recognition for the company.
In November 2021, HealthPro Innovations formed a strategic partnership with a renowned global healthcare provider. This partnership aimed to jointly develop and distribute innovative healthcare solutions, leveraging each other's expertise and resources. This collaboration not only broadens HealthPro Innovations' reach and distribution network but also opens new market opportunities and enhances its brand reputation.
In terms of corporate governance, HealthPro Innovations has demonstrated its commitment to transparency, accountability, and stakeholder engagement. In February 2021, the company proposed and obtained approval for a three-year strategic plan at the annual shareholders' meeting. This plan focuses on expanding product offerings, entering new markets, and acquiring complementary businesses. The successful passing of these resolutions shapes the company's governance structure, providing a clear direction and driving future growth opportunities.
In March 2021, HealthPro Innovations revised its corporate governance policy to increase transparency, accountability, and stakeholder engagement. The revised policy includes provisions for regular communication with shareholders and the formation of an independent governance committee. This event positively influences corporate governance practices, fostering trust among stakeholders and promoting long-term sustainability.
The appointment of Mr. Alex Johnson as the new Chairman of the Board in April 2021 brought extensive experience and strategic vision to guide the company's growth. This appointment elevates the company's strategic direction, leveraging Mr. Johnson's expertise to further enhance shareholder value.
HealthPro Innovations has also demonstrated its commitment to ethics and integrity. In May 2021, the company swiftly responded to a minor ethics violation by conducting an internal investigation, terminating the employee involved, and reinforcing its robust code of ethics. This incident had no significant financial impact on the company and showcased its commitment to ethical practices, preserving its reputation and strengthening stakeholder trust.
To ensure sustained operational stability and growth, HealthPro Innovations enhanced its risk management framework in June 2021. The company implemented updated procedures to identify emerging risks, conducted regular risk assessments, and established an enterprise risk management committee. This event enhances the company's ability to navigate risks effectively, ensuring sustained operational stability and growth.
In August 2021, HealthPro Innovations appointed Ms. Sarah Thompson as the new Chief Financial Officer. Leveraging her extensive financial management experience, Ms. Thompson's appointment enhances the company's financial operations. This ensures sound financial decision-making and maximizes shareholder value.
HealthPro Innovations has also embraced sustainability and social responsibility initiatives. In November 2021, the company launched a comprehensive sustainability program, reducing its carbon footprint by 20% through energy-efficient manufacturing processes and optimized supply chain logistics. Additionally, HealthPro Innovations supports local healthcare charities, providing free medical services to underserved communities. These initiatives positively influence the company's public image, promoting brand loyalty and attracting socially conscious investors and partners.
HealthPro Innovations maintains a robust information disclosure system, ensuring timely and comprehensive reporting to shareholders and stakeholders. Quarterly financial reports, annual reports, and quarterly conference calls provide transparent and accurate information to investors and analysts. The company has also implemented strict policies and measures to ensure fairness and compliance in all related transactions. An independent committee reviews and approves all related party transactions, ensuring transparency and fairness in dealings with related entities. Furthermore, HealthPro Innovations has developed a comprehensive internal control system to mitigate risks and ensure effective governance. Regular internal audits, segregation of duties, and a robust system of checks and balances provide a strong control environment.
Looking ahead, HealthPro Innovations has outlined several governance improvement plans. The company aims to strengthen the function of the board of directors and supervisory board through regular training programs, increased diversity, and an improved decision-making process. Additionally, HealthPro Innovations is committed to enhancing transparency and the quality of information disclosure by adopting best practices and ensuring timely, accurate, and comprehensive reporting to all stakeholders. The company also plans to strengthen its internal control system to identify, assess, monitor, and report financial and operational risks. Furthermore, HealthPro Innovations aims to integrate sustainable development and social responsibility into its strategy, identifying related risks and opportunities and incorporating them into the company's risk management framework.
Overall, HealthPro Innovations has demonstrated its commitment to corporate governance, compliance, and ethical practices. The company's strategic partnerships, acquisitions, and innovative product launches have positioned it as a leader in the healthcare industry. With a strong focus on sustainability and social responsibility, HealthPro Innovations is well-positioned for future growth and success.

Question : "How much funding did HealthPro Innovations raise in February 2021?"

**Generated Answer:** HealthPro Innovations raised $150 million in funds in February 2021 through a large-scale financing activity.

**Used Model:** GPT-5.5


**6b)** We want to use in-context learning and provide the LLM one example of a possible answer. Please, use the same prompt and extend it, that it should follow this example answer: "Yep, they sold a lot in that year. Over 50 million units as I can see — pretty big move, respect!". Use the same model, create a fresh chat and run this new prompt. Highlight the changes in the prompt using **bold style** or <span style="color:red;">color</span>.

You are an assistant answering questions

Instructions :
-  use only information coming from the context
**- answer following the example "Yep, they sold a lot in that year. Over 50 million units as I can see — pretty big move, respect!"**

Context : HealthPro Innovations, established in September 2009, is a publicly traded healthcare company based in New York, specializing in the development and sale of innovative healthcare solutions.
In 2021, HealthPro Innovations experienced several significant events that had a profound impact on its financial performance and market position. Firstly, in February 2021, the company conducted a large-scale financing activity, raising $150 million in funds. This financing activity strengthened the company's financial strength and provided support for its expansion and development. One of the sub-events that contributed to this financing activity was the strategic partnership formed with a leading healthcare provider in January 2021. This partnership aimed to expand HealthPro Innovations' market reach and gain access to new customers, ultimately resulting in increased sales and revenue potential.
In March 2021, HealthPro Innovations successfully launched a new line of innovative healthcare solutions, offering advanced features and improved patient outcomes. This product launch stimulated customer interest and generated additional revenue streams for the company. Additionally, in April 2021, HealthPro Innovations acquired a promising research and development startup specializing in cutting-edge medical technology. This acquisition enhanced the company's research capabilities, accelerating the development of new and improved healthcare solutions.
Furthermore, in June 2021, HealthPro Innovations expanded its operations into key international markets, establishing partnerships with local distributors and healthcare providers. This expansion increased the company's market share, revenue, and global presence. In August 2021, the company received an industry award for its outstanding innovation and contribution to the healthcare sector. This award enhanced HealthPro Innovations' brand reputation, attracting potential customers, investors, and partners.
In September 2021, HealthPro Innovations achieved a major research breakthrough, resulting in a significant advancement in the effectiveness and efficiency of its healthcare solutions. This breakthrough propelled the company ahead of competitors, increased market demand, and had the potential for higher sales. In November 2021, HealthPro Innovations obtained a crucial regulatory approval for one of its flagship products, allowing for expanded market access and increased sales potential. This approval boosted confidence among customers and investors, driving sales growth and enhancing the company's market position.
These events had a direct impact on several financial indicators for HealthPro Innovations in 2021. The company reported an operating income of $250 million, influenced by market demand and changes in product prices. The net profit for the year was $40 million, influenced by the operating income, cost control, and non-recurring gains and losses. HealthPro Innovations' total assets stood at $500 million, influenced by asset acquisition, disposal, and revaluation. The company's total liabilities amounted to $200 million, influenced by new debt issuance, debt repayment, and debt restructuring. Shareholder equity reached $300 million, influenced by the net profit, dividend distribution, and capital reserves. The company generated a cash flow of $80 million, influenced by operating activities, investment activities, and financing activities. The debt ratio was 40%, reflecting the company's debt level, while the debt to assets ratio was 0.4, indicating the company's financial leverage. The return on equity was 13.33%, reflecting the operational efficiency of shareholder equity.
Looking ahead, HealthPro Innovations aims to implement a comprehensive financial strategy focused on cost control and optimizing capital operations to achieve sustainable growth and profitability. The company plans to invest heavily in research and development of new healthcare solutions, expand into emerging markets, and enhance its manufacturing capabilities. HealthPro Innovations has identified major financial risks such as changes in healthcare regulations, intense competition, and currency fluctuations. The company will closely monitor these risks and implement appropriate risk management strategies to mitigate their impact on financial performance.
In conclusion, HealthPro Innovations has experienced significant events in 2021 that have shaped its financial performance and market position. Through strategic partnerships, product launches, acquisitions, international expansion, and regulatory approvals, the company has strengthened its presence in the healthcare industry. With a focus on financial strategy, investment plans, and risk management, HealthPro Innovations is well-positioned for future growth and success.
The company has been listed on the New York Stock Exchange since its establishment, positioning itself as a key player in the healthcare industry. HealthPro Innovations operates in an industry that is constantly evolving, and as such, it has demonstrated its commitment to compliance and regulatory updates.
In January 2021, HealthPro Innovations took proactive steps to comply with new regulations, particularly in the area of data protection. The company implemented enhanced data protection measures in accordance with updated healthcare data privacy laws. This event ensures that HealthPro Innovations remains compliant with evolving laws, safeguarding sensitive data and protecting the interests of the company and its shareholders.
In March 2021, HealthPro Innovations made a strategic move to acquire MedTech Solutions, a leading provider of medical technology software solutions. This acquisition aimed to expand the company's portfolio of healthcare solutions and enhance its market position. The acquisition of MedTech Solutions not only increases HealthPro Innovations' market share but also broadens its product offerings, leading to potential revenue growth and improved competitiveness in the healthcare industry.
In July 2021, HealthPro Innovations made a significant breakthrough in the field of diagnostic technology. The company launched a state-of-the-art AI-powered diagnostic device that revolutionizes the accuracy and efficiency of medical diagnoses. This innovative product attracted significant attention from healthcare providers and professionals, positioning HealthPro Innovations as an industry leader in diagnostic technology. This event has the potential to increase sales and market recognition for the company.
In November 2021, HealthPro Innovations formed a strategic partnership with a renowned global healthcare provider. This partnership aimed to jointly develop and distribute innovative healthcare solutions, leveraging each other's expertise and resources. This collaboration not only broadens HealthPro Innovations' reach and distribution network but also opens new market opportunities and enhances its brand reputation.
In terms of corporate governance, HealthPro Innovations has demonstrated its commitment to transparency, accountability, and stakeholder engagement. In February 2021, the company proposed and obtained approval for a three-year strategic plan at the annual shareholders' meeting. This plan focuses on expanding product offerings, entering new markets, and acquiring complementary businesses. The successful passing of these resolutions shapes the company's governance structure, providing a clear direction and driving future growth opportunities.
In March 2021, HealthPro Innovations revised its corporate governance policy to increase transparency, accountability, and stakeholder engagement. The revised policy includes provisions for regular communication with shareholders and the formation of an independent governance committee. This event positively influences corporate governance practices, fostering trust among stakeholders and promoting long-term sustainability.
The appointment of Mr. Alex Johnson as the new Chairman of the Board in April 2021 brought extensive experience and strategic vision to guide the company's growth. This appointment elevates the company's strategic direction, leveraging Mr. Johnson's expertise to further enhance shareholder value.
HealthPro Innovations has also demonstrated its commitment to ethics and integrity. In May 2021, the company swiftly responded to a minor ethics violation by conducting an internal investigation, terminating the employee involved, and reinforcing its robust code of ethics. This incident had no significant financial impact on the company and showcased its commitment to ethical practices, preserving its reputation and strengthening stakeholder trust.
To ensure sustained operational stability and growth, HealthPro Innovations enhanced its risk management framework in June 2021. The company implemented updated procedures to identify emerging risks, conducted regular risk assessments, and established an enterprise risk management committee. This event enhances the company's ability to navigate risks effectively, ensuring sustained operational stability and growth.
In August 2021, HealthPro Innovations appointed Ms. Sarah Thompson as the new Chief Financial Officer. Leveraging her extensive financial management experience, Ms. Thompson's appointment enhances the company's financial operations. This ensures sound financial decision-making and maximizes shareholder value.
HealthPro Innovations has also embraced sustainability and social responsibility initiatives. In November 2021, the company launched a comprehensive sustainability program, reducing its carbon footprint by 20% through energy-efficient manufacturing processes and optimized supply chain logistics. Additionally, HealthPro Innovations supports local healthcare charities, providing free medical services to underserved communities. These initiatives positively influence the company's public image, promoting brand loyalty and attracting socially conscious investors and partners.
HealthPro Innovations maintains a robust information disclosure system, ensuring timely and comprehensive reporting to shareholders and stakeholders. Quarterly financial reports, annual reports, and quarterly conference calls provide transparent and accurate information to investors and analysts. The company has also implemented strict policies and measures to ensure fairness and compliance in all related transactions. An independent committee reviews and approves all related party transactions, ensuring transparency and fairness in dealings with related entities. Furthermore, HealthPro Innovations has developed a comprehensive internal control system to mitigate risks and ensure effective governance. Regular internal audits, segregation of duties, and a robust system of checks and balances provide a strong control environment.
Looking ahead, HealthPro Innovations has outlined several governance improvement plans. The company aims to strengthen the function of the board of directors and supervisory board through regular training programs, increased diversity, and an improved decision-making process. Additionally, HealthPro Innovations is committed to enhancing transparency and the quality of information disclosure by adopting best practices and ensuring timely, accurate, and comprehensive reporting to all stakeholders. The company also plans to strengthen its internal control system to identify, assess, monitor, and report financial and operational risks. Furthermore, HealthPro Innovations aims to integrate sustainable development and social responsibility into its strategy, identifying related risks and opportunities and incorporating them into the company's risk management framework.
Overall, HealthPro Innovations has demonstrated its commitment to corporate governance, compliance, and ethical practices. The company's strategic partnerships, acquisitions, and innovative product launches have positioned it as a leader in the healthcare industry. With a strong focus on sustainability and social responsibility, HealthPro Innovations is well-positioned for future growth and success.

Question : "How much funding did HealthPro Innovations raise in February 2021?"

**Generated Answer:** "Yep, they raised a solid amount in February 2021 — $150 million in funding as I can see. Big financial boost, that really strengthened their expansion plans."

**6c)** Please check if the two answers are correct according to the document and how they differ. Does the model follow the example in the second prompt?

**Your Answer:** they state the correct information that is provided in the document. The second answer differs by praising a bit the company, it does sound more humane too. They accurately followed the example in the second prompt

## End of AdvNLP Lab 4

Please make sure all cells have been executed, save this completed notebook, and upload it to Moodle.